# Notebook 7 - Closing the Remaining Reviewer Requests
### Scientific Reports 35109da0

Three outstanding items, all of which need the live registries:

| part | reviewer request | your time |
|---|---|---|
| **A** | real-incident case study (R1-m4) | ~20 min review |
| **B** | manual verification of the positive class (R1-2, R3-1) | 3-5 h |
| **C** | end-to-end runtime measurements (R2-8) | ~20 min, unattended |

Part B is the long one and it is not optional: three reviewers asked whether the positive
class actually represents update-compromise rather than malware-from-origin or typosquatting.
The tool below writes a self-contained HTML sheet you can work through in a browser, saving
after every decision.

Run parts in any order.

## Setup

In [ ]:
!pip install esprima -q
import os, io, re, json, time, math, difflib, tarfile, zipfile, warnings, html as _html
from collections import Counter
warnings.filterwarnings("ignore")
import requests, numpy as np, pandas as pd

try:
    from google.colab import drive
    if not os.path.ismount("/content/drive"): drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/paper_revision"
except Exception:
    WORK = os.path.abspath("./paper_revision")
os.makedirs(WORK, exist_ok=True)
P = lambda n: os.path.join(WORK, n)
def log(m): print(f"[{time.strftime('%H:%M:%S')}] {m}", flush=True)

KEY = {"package":str,"ecosystem":str,"version_before":str,"version_after":str}
clean = pd.read_csv(P("features_v6_clean.csv"), dtype=KEY)
clean["label"] = clean.label.astype(int)
mal = clean[clean.label == 1].copy()
log(f"malicious pairs available: {len(mal)} across {mal.package.nunique()} packages")

## Registry and diff helpers

In [ ]:
HEADERS = {"User-Agent": "supply-chain-research/1.0 (academic)"}
DELAY, RETRIES = 0.4, 3

def _req(url, want_json, timeout):
    for a in range(RETRIES):
        try:
            r = requests.get(url, headers=HEADERS, timeout=timeout); time.sleep(DELAY)
            if r.status_code == 200: return r.json() if want_json else r.content
            if r.status_code in (429,500,502,503): time.sleep(2**a); continue
            return None
        except requests.RequestException: time.sleep(2**a)
        except ValueError: return None
    return None

def tarball_url(pkg, eco, v):
    if eco == "npm":
        m = _req(f"https://registry.npmjs.org/{pkg}", True, 15)
        try: return m["versions"][v]["dist"]["tarball"]
        except (KeyError, TypeError): return None
    m = _req(f"https://pypi.org/pypi/{pkg}/json", True, 15)
    try:
        fs = m["releases"].get(v, [])
        if not fs: return None
        sd = [f for f in fs if f["url"].endswith((".tar.gz",".zip"))]
        return (sd[0] if sd else fs[0])["url"]
    except (KeyError, TypeError): return None

def strip_root(n):
    p = n.split("/", 1); return p[1] if len(p) > 1 else n

def extract_map(arch, eco):
    if not arch: return {}
    exts = (".js",".ts",".json",".mjs",".cjs") if eco=="npm" else (".py",".cfg",".toml",".txt")
    out = {}
    try:
        try:
            with tarfile.open(fileobj=io.BytesIO(arch), mode="r:gz") as t:
                for m in t.getmembers():
                    if m.isfile() and m.name.endswith(exts) and m.size < 2_000_000:
                        f = t.extractfile(m)
                        if f: out[strip_root(m.name)] = f.read().decode("utf-8","ignore")
        except tarfile.ReadError:
            with zipfile.ZipFile(io.BytesIO(arch)) as z:
                for n in z.namelist():
                    if n.endswith(exts):
                        try: out[strip_root(n)] = z.read(n).decode("utf-8","ignore")
                        except Exception: pass
    except Exception: return {}
    return out

def unified_diff(fb, fa, max_lines=400):
    """Readable diff of the update, newest files first."""
    chunks = []
    for path in sorted(set(fa) | set(fb)):
        b, a = fb.get(path, ""), fa.get(path, "")
        if b == a: continue
        d = list(difflib.unified_diff(b.splitlines(), a.splitlines(),
                                      fromfile=f"a/{path}", tofile=f"b/{path}", n=2))
        if d: chunks.append("\n".join(d[:max_lines]))
    return "\n\n".join(chunks)[:60000]

def pair_diff(pkg, eco, vb, va):
    fa = extract_map(_req(tarball_url(pkg, eco, va), False, 30), eco)
    fb = extract_map(_req(tarball_url(pkg, eco, vb), False, 30), eco)
    if not fa: return None, None, None
    return unified_diff(fb, fa), fb, fa

log("helpers ready")

---
# Part A - real-incident case study

Reviewer 1 asked for the illustrative example to be replaced with a documented incident from
the dataset, showing the code change, the extracted features, the model score, and an analyst
reading of it.

Candidates are ranked by evidence strength: a taint chain present, and a diff small enough to
print in the paper. **Read the printed diff and confirm it is genuinely malicious** before it
goes into the manuscript.

In [ ]:
sec_cols = [c for c in mal.columns if c.startswith("add_")]
cand = mal[(mal.added_lines.between(5, 300)) & (mal.tb_chain_blocks > 0)].copy()
cand = cand.sort_values(["tb_chain_blocks","add_env_cred_access"], ascending=False)
cols = ["package","ecosystem","version_before","version_after","added_lines",
        "add_env_cred_access","add_net_http","add_encoded_blob","tb_chain_blocks"]
log(f"candidates: {len(cand)}")
print(cand[cols].head(10).to_string(index=False))

In [ ]:
# print the full diff of the top candidates so you can pick one
N_SHOW = 3
for i, r in enumerate(cand.head(N_SHOW).itertuples(index=False), 1):
    print("=" * 100)
    print(f"CANDIDATE {i}: {r.package} ({r.ecosystem})  {r.version_before} -> {r.version_after}")
    print(f"  added_lines={r.added_lines}  cred={r.add_env_cred_access}  "
          f"net={r.add_net_http}  encoded={r.add_encoded_blob}  chains={r.tb_chain_blocks}")
    print("=" * 100)
    d, _, _ = pair_diff(r.package, r.ecosystem, str(r.version_before), str(r.version_after))
    print(d[:6000] if d else "  (archive unavailable)")
    print()

In [ ]:
# once you have chosen, set CASE and this writes the case-study record for the paper
CASE = None      # e.g. ("node-env-resolver-dotenvx", "npm", "2.0.0", "1.0.1")

if CASE:
    pkg, eco, vb, va = CASE
    row = mal[(mal.package==pkg)&(mal.version_before==vb)&(mal.version_after==va)]
    d, fb, fa = pair_diff(pkg, eco, vb, va)
    rec = {"package":pkg, "ecosystem":eco, "version_before":vb, "version_after":va,
           "diff": d,
           "features": {c: (row[c].iloc[0].item() if hasattr(row[c].iloc[0],'item')
                            else row[c].iloc[0])
                        for c in cols[4:] + ["tb_encoded_flow","sc_toplevel_stmts",
                                             "ih_install_hook_added"] if c in row.columns}}
    json.dump(rec, open(P("case_study.json"), "w"), indent=2, default=str)
    log(f"saved case_study.json for {pkg} {vb} -> {va}")
    print(json.dumps(rec["features"], indent=2))

---
# Part B - manual verification of the positive class

> R1-2: *"provide evidence that the preceding release is benign, state explicit exclusion
> rules"*
> R3-1: *"perform incident-level verification, exclude first-release malware, typosquatting,
> dependency-confusion, and repeated-malicious-version cases"*

### What runs automatically

Four exclusion rules are applied first, so you only review what the rules cannot settle:

| rule | meaning |
|---|---|
| `first_release` | the flagged version is the package's first - malware-from-origin, not an update |
| `repeated_malicious` | several flagged versions in a row - the predecessor may also be malicious |
| `typosquat_suspect` | name within edit distance 2 of a popular package |
| `empty_diff` | nothing added, nothing to judge |

### What you do

The cell after that writes **`verification_sheet.html`**. Download it, open it in a browser,
and classify each pair. Progress saves to browser storage after every click and the sheet
exports a CSV when you are done. Budget 1-2 minutes per pair.

In [ ]:
POPULAR = {
 "npm": ["lodash","react","express","axios","chalk","commander","debug","moment","request",
         "async","underscore","jquery","webpack","babel","eslint","typescript","dotenv",
         "colors","uuid","yargs","semver","glob","minimist","mkdirp","rimraf","cross-env"],
 "pypi": ["requests","urllib3","numpy","pandas","flask","django","setuptools","six","click",
          "jinja2","pyyaml","boto3","pytest","scipy","pillow","cryptography","certifi",
          "colorama","python-dateutil","beautifulsoup4","matplotlib","tqdm"],
}

def lev(a, b):
    if abs(len(a)-len(b)) > 2: return 9
    prev = list(range(len(b)+1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j]+1, cur[j-1]+1, prev[j-1] + (ca != cb)))
        prev = cur
    return prev[-1]

def version_index(pkg, eco, v):
    if eco == "npm":
        m = _req(f"https://registry.npmjs.org/{pkg}", True, 15)
        if not m: return None, None
        t = m.get("time", {}) or {}
        vs = sorted([x for x in (m.get("versions") or {}) if t.get(x)], key=lambda x: t[x])
    else:
        m = _req(f"https://pypi.org/pypi/{pkg}/json", True, 15)
        if not m: return None, None
        rel = m.get("releases", {}) or {}
        vs = sorted([k for k, f in rel.items() if f],
                    key=lambda k: rel[k][0].get("upload_time_iso_8601", ""))
    return (vs.index(v) if v in vs else None), len(vs)

CK_RULES = P("exclusion_rules.csv")
mal2 = mal.copy()
# R3-1 concerns pairs whose PREDECESSOR is itself flagged, which breaks the benign-baseline
# assumption. Test that directly rather than penalising any package with several incidents:
# a package with many separate compromises is still legitimate evidence.
flagged = set(zip(mal2.package, mal2.ecosystem, mal2.version_after.astype(str)))
mal2["predecessor_flagged"] = [
    int((p, e, str(v)) in flagged)
    for p, e, v in zip(mal2.package, mal2.ecosystem, mal2.version_before)]
mal2["n_flagged_in_pkg"] = mal2.groupby(["package","ecosystem"]).package.transform("size")

rows = []
done = set(pd.read_csv(CK_RULES).apply(
    lambda r: (str(r["package"]), str(r["version_after"])), axis=1)) \
    if os.path.exists(CK_RULES) else set()

for n, r in enumerate(mal2.itertuples(index=False), 1):
    if (str(r.package), str(r.version_after)) in done: continue
    idx, total = version_index(r.package, r.ecosystem, str(r.version_after))
    name = r.package.lower().split("/")[-1]
    ts = min((lev(name, p) for p in POPULAR.get(r.ecosystem, [])), default=9)
    rows.append({
        "package": r.package, "ecosystem": r.ecosystem,
        "version_before": r.version_before, "version_after": r.version_after,
        "version_index": -1 if idx is None else idx, "n_versions": total or -1,
        "first_release": int(idx == 0) if idx is not None else 0,
        "repeated_malicious": int(r.predecessor_flagged),
        "typosquat_suspect": int(0 < ts <= 2),
        "empty_diff": int(r.added_lines == 0),
    })
    if n % 25 == 0:
        pd.DataFrame(rows).to_csv(CK_RULES, mode="a",
                                  header=not os.path.exists(CK_RULES), index=False)
        rows = []
        log(f"  {n}/{len(mal2)} rules applied")
if rows:
    pd.DataFrame(rows).to_csv(CK_RULES, mode="a", header=not os.path.exists(CK_RULES), index=False)

rules = pd.read_csv(CK_RULES).drop_duplicates(subset=["package","version_after"])
excl = rules[["first_release","repeated_malicious","typosquat_suspect","empty_diff"]]
rules["excluded"] = (excl.sum(axis=1) > 0).astype(int)
log("automatic exclusions (these counts go into the construction funnel):")
log(str(excl.sum().to_string()))
log(f"excluded overall: {int(rules.excluded.sum())} of {len(rules)}")
log(f"(for the record: {int((mal2.n_flagged_in_pkg > 3).sum())} pairs sit in packages with "
    f">3 separate flagged versions; these are kept, since repeated independent compromises "
    f"are still valid evidence, but the count is reported)")
log(f"remaining for manual review: {int((~rules.excluded.astype(bool)).sum())}")

In [ ]:
# sample the pairs to review and fetch their diffs
N_REVIEW = 120
review = rules[rules.excluded == 0].sample(
    min(N_REVIEW, int((rules.excluded == 0).sum())), random_state=42)
log(f"fetching diffs for {len(review)} pairs (~{len(review)*3//60} min)")

items = []
for n, r in enumerate(review.itertuples(index=False), 1):
    d, _, _ = pair_diff(r.package, r.ecosystem, str(r.version_before), str(r.version_after))
    items.append({"package": r.package, "ecosystem": r.ecosystem,
                  "version_before": r.version_before, "version_after": r.version_after,
                  "diff": (d or "(archive unavailable)")[:20000]})
    if n % 20 == 0: log(f"  {n}/{len(review)}")
json.dump(items, open(P("review_items.json"), "w"))
log(f"saved review_items.json ({len(items)} pairs)")

In [ ]:
ITEMS_JSON = json.dumps(items)
HTML = """<!doctype html><html><head><meta charset="utf-8"><title>Pair verification</title>
<style>
body{font:14px/1.5 -apple-system,Segoe UI,Roboto,sans-serif;margin:0;background:#f6f7f9;color:#111}
header{position:sticky;top:0;background:#fff;border-bottom:1px solid #ddd;padding:12px 20px;
 display:flex;gap:16px;align-items:center;flex-wrap:wrap;z-index:9}
h1{font-size:16px;margin:0;font-weight:600}
.meta{font-family:ui-monospace,Menlo,monospace;font-size:13px;color:#444}
.wrap{padding:16px 20px;max-width:1100px;margin:0 auto}
pre{background:#fff;border:1px solid #ddd;border-radius:6px;padding:12px;overflow:auto;
 max-height:60vh;font-family:ui-monospace,Menlo,monospace;font-size:12px;white-space:pre-wrap}
.add{background:#e6ffed}.del{background:#ffeef0}.hdr{color:#666;font-weight:600}
button{font:14px inherit;padding:9px 14px;border-radius:6px;border:1px solid #ccc;
 background:#fff;cursor:pointer}
button:hover{background:#f0f0f0}
.b1{border-color:#c00;color:#c00}.b2{border-color:#06c;color:#06c}
.b3{border-color:#a60;color:#a60}.b4{border-color:#666;color:#666}
#bar{height:4px;background:#06c;width:0;transition:width .2s}
</style></head><body>
<div id="bar"></div>
<header>
  <h1>Update-compromise verification</h1>
  <span class="meta" id="pos"></span>
  <span class="meta" id="pkg"></span>
  <span style="flex:1"></span>
  <button onclick="back()">&larr; back</button>
  <button onclick="exportCsv()">export CSV</button>
</header>
<div class="wrap">
  <div style="display:flex;gap:8px;margin-bottom:12px;flex-wrap:wrap">
    <button class="b1" onclick="mark('update_compromise')">1 &nbsp;update compromise</button>
    <button class="b2" onclick="mark('malware_from_origin')">2 &nbsp;malware from origin</button>
    <button class="b3" onclick="mark('typosquat')">3 &nbsp;typosquat / confusion</button>
    <button class="b4" onclick="mark('unclear')">4 &nbsp;unclear</button>
  </div>
  <pre id="diff"></pre>
</div>
<script>
const ITEMS = __ITEMS__;
let i = 0, ans = JSON.parse(localStorage.getItem('verif')||'{}');
function key(it){return it.package+'|'+it.version_before+'|'+it.version_after}
function render(){
  while(i<ITEMS.length && ans[key(ITEMS[i])]) i++;
  if(i>=ITEMS.length){document.getElementById('diff').textContent=
    'All done. Click "export CSV".';document.getElementById('pos').textContent='';return;}
  const it=ITEMS[i];
  document.getElementById('pos').textContent=(i+1)+' / '+ITEMS.length+
    '  ('+Object.keys(ans).length+' done)';
  document.getElementById('pkg').textContent=it.package+' ['+it.ecosystem+']  '+
    it.version_before+' -> '+it.version_after;
  document.getElementById('bar').style.width=(100*Object.keys(ans).length/ITEMS.length)+'%';
  const pre=document.getElementById('diff');pre.innerHTML='';
  it.diff.split('\\n').forEach(l=>{
    const s=document.createElement('span');
    s.textContent=l+'\\n';
    if(l.startsWith('+')&&!l.startsWith('+++'))s.className='add';
    else if(l.startsWith('-')&&!l.startsWith('---'))s.className='del';
    else if(l.startsWith('@@')||l.startsWith('---')||l.startsWith('+++'))s.className='hdr';
    pre.appendChild(s);
  });
  pre.scrollTop=0;
}
function mark(v){ans[key(ITEMS[i])]=v;localStorage.setItem('verif',JSON.stringify(ans));i++;render()}
function back(){if(i>0){i--;delete ans[key(ITEMS[i])];
  localStorage.setItem('verif',JSON.stringify(ans));render()}}
function exportCsv(){
  let out='package,ecosystem,version_before,version_after,verdict\\n';
  ITEMS.forEach(it=>{const v=ans[key(it)];if(v)out+=[it.package,it.ecosystem,
    it.version_before,it.version_after,v].join(',')+'\\n';});
  const a=document.createElement('a');
  a.href=URL.createObjectURL(new Blob([out],{type:'text/csv'}));
  a.download='manual_verification.csv';a.click();
}
document.addEventListener('keydown',e=>{
  if(['1','2','3','4'].includes(e.key))
    mark(['update_compromise','malware_from_origin','typosquat','unclear'][+e.key-1]);
  if(e.key==='Backspace')back();
});
render();
</script></body></html>""".replace("__ITEMS__", ITEMS_JSON)

open(P("verification_sheet.html"), "w", encoding="utf-8").write(HTML)
log(f"saved verification_sheet.html ({len(HTML)//1024} KB)")
log("Download it, open in a browser, press 1-4 per pair (Backspace undoes).")
try:
    from google.colab import files
    files.download(P("verification_sheet.html"))
except Exception: pass

In [ ]:
# after you export manual_verification.csv, upload it and run this for the paper numbers
MV = P("manual_verification.csv")
if os.path.exists(MV):
    mv = pd.read_csv(MV)
    vc = mv.verdict.value_counts()
    n = len(mv); k = int(vc.get("update_compromise", 0))
    p = k / n
    se = math.sqrt(p*(1-p)/n)
    log(f"reviewed: {n}")
    log(str(vc.to_string()))
    log(f"update-compromise rate: {p:.3f}  95% CI [{max(0,p-1.96*se):.3f}, {min(1,p+1.96*se):.3f}]")
    json.dump({"n_reviewed": n, "counts": vc.to_dict(), "rate": round(p,4),
               "ci95": [round(max(0,p-1.96*se),4), round(min(1,p+1.96*se),4)]},
              open(P("manual_verification_summary.json"), "w"), indent=2)
else:
    log("manual_verification.csv not found yet - export it from the HTML sheet first, "
        "then upload it to the Drive folder and re-run this cell.")

---
# Part C - end-to-end runtime

> R2-8: *"report empirical end-to-end measurements: archive download time, feature-extraction
> time, inference time, memory consumption, throughput, and failure rates for npm and PyPI
> separately"*

Each stage is timed separately on a random sample, so the claim that network retrieval
dominates is measured rather than asserted.

In [ ]:
import tracemalloc
N_TIME = 40
sample = clean.sample(min(N_TIME, len(clean)), random_state=42)
recs = []
tracemalloc.start()
for n, r in enumerate(sample.itertuples(index=False), 1):
    rec = {"ecosystem": r.ecosystem, "ok": 0}
    t0 = time.perf_counter()
    u = tarball_url(r.package, r.ecosystem, str(r.version_after))
    t1 = time.perf_counter()
    arch = _req(u, False, 30) if u else None
    t2 = time.perf_counter()
    files = extract_map(arch, r.ecosystem) if arch else {}
    t3 = time.perf_counter()
    rec.update(metadata_s=t1-t0, download_s=t2-t1, extract_s=t3-t2,
               archive_kb=(len(arch)/1024 if arch else 0), n_files=len(files),
               ok=int(bool(files)))
    recs.append(rec)
    if n % 10 == 0: log(f"  {n}/{len(sample)}")
cur, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()

t = pd.DataFrame(recs)
log(f"peak memory during extraction: {peak/1e6:.1f} MB")
log(f"failure rate: {1-t.ok.mean():.3f}")
summ = t[t.ok==1].groupby("ecosystem")[
    ["metadata_s","download_s","extract_s","archive_kb","n_files"]].agg(["median","mean"])
log("per-stage timings (seconds):")
print(summ.round(3).to_string())

ok = t[t.ok==1]
tot = ok.metadata_s + ok.download_s + ok.extract_s
log(f"\nend-to-end per pair: median {tot.median():.2f}s  mean {tot.mean():.2f}s")
log(f"network share of runtime: "
    f"{100*(ok.metadata_s+ok.download_s).sum()/max(tot.sum(),1e-9):.1f}%")
log(f"throughput: {3600/max(tot.mean(),1e-9):.0f} candidate releases per hour, single-threaded")

# model inference cost, separately
from sklearn.ensemble import HistGradientBoostingClassifier
F = [c for c in clean.columns if c.startswith(("add_","rem_","net_","addrate_","tb_","sc_","ih_"))]
X = clean[F].fillna(0).values
m = HistGradientBoostingClassifier(max_iter=300, random_state=42).fit(X, clean.label)
t0 = time.perf_counter()
for _ in range(20): m.predict_proba(X)
inf = (time.perf_counter()-t0)/(20*len(X))
log(f"inference: {inf*1e6:.1f} microseconds per candidate")

json.dump({"peak_memory_mb": round(peak/1e6,1), "failure_rate": round(1-t.ok.mean(),4),
           "median_end_to_end_s": round(float(tot.median()),3),
           "network_share_pct": round(float(100*(ok.metadata_s+ok.download_s).sum()/max(tot.sum(),1e-9)),1),
           "throughput_per_hour": int(3600/max(tot.mean(),1e-9)),
           "inference_us": round(inf*1e6,2),
           "by_ecosystem": json.loads(summ.round(4).to_json())},
          open(P("runtime_measurements.json"), "w"), indent=2)
log("saved runtime_measurements.json")

---
## What to send back

| file | for |
|---|---|
| `case_study.json` | Part A - the worked example |
| `manual_verification_summary.json` | Part B - the verification rate and CI |
| `exclusion_rules.csv` | Part B - funnel counts for the construction figure |
| `runtime_measurements.json` | Part C - the deployment-cost table |

With these, every reviewer request is either satisfied or explicitly answered.